In [8]:
import os
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.metrics import cohen_kappa_score

# load the annotation files
ANNOTATOR1_DIR = "../annotations/annotator1"
ANNOTATOR2_DIR = "../annotations/annotator2"
ANNOTATOR3_DIR = "../annotations/annotator3"
OUTPUT_DIR = "../annotations/gt"

COLUMN_INDEX = 4  # Column E, keep paraphrase T/F
ERROR_HIERARCHY = ["wrong_modif", "meaning", "realism"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [9]:
# helper functions

# match files by modification type + model
def get_file_map(folder, prefix):
    files = [f for f in os.listdir(folder) if f.endswith(".xlsx") and f.startswith(prefix)]
    return {f[len(prefix):]: os.path.join(folder, f) for f in files}

# make sure labels are the same
def normalize_labels(series):
    mapping = {
        "TRUE": True, "T": True, "1": True, "YES": True,
        "FALSE": False, "F": False, "0": False, "NO": False
    }
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .map(mapping)   
        .infer_objects(copy=False)
    )

# for 3 annotators
def fleiss_kappa(table):
    """Compute Fleiss' kappa given an (N x k) matrix of vote counts."""
    table = np.asarray(table, dtype=float)
    N, k = table.shape
    n_annotators = table.sum(axis=1)
    if not np.all(n_annotators == n_annotators[0]):
        raise ValueError("All rows must have the same number of annotators")
    n = n_annotators[0]

    p = table.sum(axis=0) / (N * n)
    P = (table * (table - 1)).sum(axis=1) / (n * (n - 1))
    P_bar = P.mean()
    P_e = (p ** 2).sum()
    return (P_bar - P_e) / (1 - P_e)

In [10]:
# check to see that everything is filled
ERROR_COLS = ["wrong_modif", "meaning", "realism"]

def validate_annotations_for_all(a1_files, a2_files, a3_files, common_suffixes):
    """Validate all annotator files and collect missing/incomplete cases."""
    issues = []  # collect dictionaries for df later

    for suffix in common_suffixes:
        for annot_id, file_dict in zip(["a1", "a2", "a3"], [a1_files, a2_files, a3_files]):
            fpath = file_dict[suffix]
            df = pd.read_excel(fpath)
            fname = os.path.basename(fpath)

            tf_col = df.columns[COLUMN_INDEX]

            # T/F values
            mask_missing_tf = df[tf_col].isna() | (df[tf_col].astype(str).str.strip() == "")
            for idx in df[mask_missing_tf].index:
                issues.append({
                    "file": fname,
                    "annotator": annot_id,
                    "row": idx + 2,  # +2 for Excel row numbering
                    "issue": "Missing T/F value",
                    "tf_value": None,
                    "wrong_modif": df.loc[idx, "wrong_modif"] if "wrong_modif" in df.columns else None,
                    "meaning": df.loc[idx, "meaning"] if "meaning" in df.columns else None,
                    "realism": df.loc[idx, "realism"] if "realism" in df.columns else None,
                })

            # error values
            for i, row in df.iterrows():
                tf_val = str(row[tf_col]).strip().upper()
                if tf_val in ["F", "FALSE"]:
                    filled_errors = sum(
                        pd.notna(row.get(c)) and str(row.get(c)).strip() != "" for c in ERROR_COLS
                    )
                    if filled_errors == 0:
                        issues.append({
                            "file": fname,
                            "annotator": annot_id,
                            "row": i + 2,
                            "issue": "F row missing error annotation",
                            "tf_value": row[tf_col],
                            "wrong_modif": row.get("wrong_modif", None),
                            "meaning": row.get("meaning", None),
                            "realism": row.get("realism", None),
                        })

            print(f"{fname}: checked {len(df)} rows")

    # convert to DataFrame
    issues_df = pd.DataFrame(issues)
    report_path = os.path.join(OUTPUT_DIR, "validation_report.csv")

    if not issues_df.empty:
        issues_df.to_csv(report_path, index=False)
        print(f"\n Validation completed with {len(issues_df)} issues found.")
        print(f"Saved detailed report → {report_path}")
    else:
        print("\n All annotation files passed validation with no issues found!")

    return issues_df

In [11]:
# match files
a1_files = get_file_map(ANNOTATOR1_DIR, "a1_")
a2_files = get_file_map(ANNOTATOR2_DIR, "a2_")
a3_files = get_file_map(ANNOTATOR3_DIR, "a3_")

common_suffixes = sorted(set(a1_files.keys()) & set(a2_files.keys()) & set(a3_files.keys()))

if not common_suffixes:
    raise ValueError("No matching files found across annotators 1, 2, and 3.")

print(f"Found {len(common_suffixes)} matching files:")
for suffix in common_suffixes:
    print(f"  - {suffix}")

Found 10 matching files:
  - AAE_chatgpt.xlsx
  - AAE_deepseek.xlsx
  - change_voice_chatgpt.xlsx
  - change_voice_deepseek.xlsx
  - formal_chatgpt.xlsx
  - formal_deepseek.xlsx
  - prepositions_chatgpt.xlsx
  - prepositions_deepseek.xlsx
  - synonym_substitution_chatgpt.xlsx
  - synonym_substitution_deepseek.xlsx


In [13]:
# validate
# Run validation across all common files
issues_df = validate_annotations_for_all(a1_files, a2_files, a3_files, common_suffixes)

# Display a preview in the notebook
if not issues_df.empty:
    display(issues_df.head())

a1_AAE_chatgpt.xlsx: checked 133 rows
a2_AAE_chatgpt.xlsx: checked 133 rows
a3_AAE_chatgpt.xlsx: checked 133 rows
a1_AAE_deepseek.xlsx: checked 538 rows
a2_AAE_deepseek.xlsx: checked 538 rows
a3_AAE_deepseek.xlsx: checked 538 rows
a1_change_voice_chatgpt.xlsx: checked 355 rows
a2_change_voice_chatgpt.xlsx: checked 355 rows
a3_change_voice_chatgpt.xlsx: checked 355 rows
a1_change_voice_deepseek.xlsx: checked 597 rows
a2_change_voice_deepseek.xlsx: checked 597 rows
a3_change_voice_deepseek.xlsx: checked 597 rows
a1_formal_chatgpt.xlsx: checked 544 rows
a2_formal_chatgpt.xlsx: checked 544 rows
a3_formal_chatgpt.xlsx: checked 544 rows
a1_formal_deepseek.xlsx: checked 561 rows
a2_formal_deepseek.xlsx: checked 561 rows
a3_formal_deepseek.xlsx: checked 561 rows
a1_prepositions_chatgpt.xlsx: checked 147 rows
a2_prepositions_chatgpt.xlsx: checked 147 rows
a3_prepositions_chatgpt.xlsx: checked 147 rows
a1_prepositions_deepseek.xlsx: checked 377 rows
a2_prepositions_deepseek.xlsx: checked 377 row

In [14]:
WRONG_MODIF_IDX = ord('J') - ord('A')   # 9
REALISM_IDX    = ord('K') - ord('A')   # 10
MEANING_IDX    = ord('L') - ord('A')   # 11

def is_marked(cell):
    """Return True if an error cell should be considered marked."""
    if pd.isna(cell):
        return False
    s = str(cell).strip().lower()
    return s not in ("", "0", "false", "f", "no", "nan")

for suffix in common_suffixes:
    df1 = pd.read_excel(a1_files[suffix])
    df2 = pd.read_excel(a2_files[suffix])
    df3 = pd.read_excel(a3_files[suffix])

    col_name = df1.columns[COLUMN_INDEX]  # T/F column
    col1 = normalize_labels(df1.iloc[:, COLUMN_INDEX])
    col2 = normalize_labels(df2.iloc[:, COLUMN_INDEX])
    col3 = normalize_labels(df3.iloc[:, COLUMN_INDEX])

    # cohen's kappa between a1 and a2
    valid_idx = [i for i in range(len(col1)) if not pd.isna(col1[i]) and not pd.isna(col2[i])]
    if valid_idx:
        kappa_tf = cohen_kappa_score([col1[i] for i in valid_idx],
                                     [col2[i] for i in valid_idx])
    else:
        kappa_tf = np.nan

    # creating gt files
    # consensus t/f using a3 as tiebreaker
    consensus_TF = []
    for v1, v2, v3 in zip(col1, col2, col3):
        votes = [v for v in (v1, v2, v3) if not pd.isna(v)]
        if len(votes) == 0:
            consensus_TF.append("")  # fully missing
            continue
        t_count = sum(1 for v in votes if v is True)
        f_count = sum(1 for v in votes if v is False)
        if t_count > f_count:
            consensus_TF.append("T")
        elif f_count > t_count:
            consensus_TF.append("F")
        else:
            # tie -> conservative default 'F'
            consensus_TF.append("F")

    # consensus error only for false rows
    consensus_error = []
    for i, tf in enumerate(consensus_TF):
        if tf == "F":
            # check across annotators in hierarchy order
            if any(is_marked(df.iloc[i, WRONG_MODIF_IDX]) for df in (df1, df2, df3)):
                consensus_error.append("wrong_modif")
            elif any(is_marked(df.iloc[i, MEANING_IDX]) for df in (df1, df2, df3)):
                consensus_error.append("meaning")
            elif any(is_marked(df.iloc[i, REALISM_IDX]) for df in (df1, df2, df3)):
                consensus_error.append("realism")
            else:
                consensus_error.append("")  # no error marked by any annotator (shouldn't happen if validated)
        else:
            consensus_error.append("")  # for 'T' rows

    # replacing columns directly in df1
    df1[col_name] = consensus_TF
    
    for col in ["uncertain", "wrong_modif", "realism", "meaning"]: # clear this first
        if col in df1.columns:
            df1[col] = ""

    nrows = len(df1)
    # prepare lists of values
    wrong_vals = [1 if e == "wrong_modif" else "" for e in consensus_error]
    realism_vals = [1 if e == "realism" else "" for e in consensus_error]
    meaning_vals = [1 if e == "meaning" else "" for e in consensus_error]

    # assign by iloc if the columns exist at those indices; fallback: try by header name
    if WRONG_MODIF_IDX < len(df1.columns):
        df1.iloc[:, WRONG_MODIF_IDX] = wrong_vals
    elif "wrong_modif" in df1.columns:
        df1["wrong_modif"] = wrong_vals

    if REALISM_IDX < len(df1.columns):
        df1.iloc[:, REALISM_IDX] = realism_vals
    elif "realism" in df1.columns:
        df1["realism"] = realism_vals

    if MEANING_IDX < len(df1.columns):
        df1.iloc[:, MEANING_IDX] = meaning_vals
    elif "meaning" in df1.columns:
        df1["meaning"] = meaning_vals
        
    # save output
    output_path = os.path.join(OUTPUT_DIR, f"annotated_{suffix}")
    df1.to_excel(output_path, index=False)

    print(f"{suffix}: Cohen's Kappa (T/F) = {kappa_tf:.3f}")
    print(f"Saved → {output_path}")

AAE_chatgpt.xlsx: Cohen's Kappa (T/F) = 0.451
Saved → ../annotations/gt/annotated_AAE_chatgpt.xlsx
AAE_deepseek.xlsx: Cohen's Kappa (T/F) = 0.589
Saved → ../annotations/gt/annotated_AAE_deepseek.xlsx
change_voice_chatgpt.xlsx: Cohen's Kappa (T/F) = 0.507
Saved → ../annotations/gt/annotated_change_voice_chatgpt.xlsx
change_voice_deepseek.xlsx: Cohen's Kappa (T/F) = 0.332
Saved → ../annotations/gt/annotated_change_voice_deepseek.xlsx
formal_chatgpt.xlsx: Cohen's Kappa (T/F) = 0.790
Saved → ../annotations/gt/annotated_formal_chatgpt.xlsx
formal_deepseek.xlsx: Cohen's Kappa (T/F) = 0.751
Saved → ../annotations/gt/annotated_formal_deepseek.xlsx
prepositions_chatgpt.xlsx: Cohen's Kappa (T/F) = 0.755
Saved → ../annotations/gt/annotated_prepositions_chatgpt.xlsx
prepositions_deepseek.xlsx: Cohen's Kappa (T/F) = 0.923
Saved → ../annotations/gt/annotated_prepositions_deepseek.xlsx
synonym_substitution_chatgpt.xlsx: Cohen's Kappa (T/F) = 0.225
Saved → ../annotations/gt/annotated_synonym_substitut